# analyze_convergence_clubs — Phillips-Sul convergence clubs

_Notebook version: built 2026-06-22 05:34 UTC — re-open this notebook from GitHub if yours is older, to get the latest version._

An extended **user guide** *and* a **verification harness** for `analyze_convergence_clubs` from [expdpy](https://github.com/cmg777/expdpy): the log(t) test, the HP-filter trend, the data-driven clustering, every argument, everything it returns, and a synthetic-data check that it recovers a planted club structure. Run the install cell below first, then run the rest top to bottom.

> The first cell installs everything and then **restarts the Colab runtime once** so the upgraded NumPy loads cleanly. When it reconnects, run the cells again (Runtime > Run all) — the install cell skips the restart the second time.

This notebook mirrors the [analyze_convergence_clubs page](https://cmg777.github.io/expdpy/analyze_convergence_clubs.html) of the docs.

In [ ]:
import importlib.util
import os

!pip install -q "numpy>=2.1" "numba>=0.61" "expdpy @ git+https://github.com/cmg777/expdpy.git"
!pip install -q --force-reinstall --no-deps "expdpy @ git+https://github.com/cmg777/expdpy.git"

_RESTART_FLAG = "/tmp/.expdpy_runtime_restarted"
_ON_COLAB = importlib.util.find_spec("google.colab") is not None
if _ON_COLAB and not os.path.exists(_RESTART_FLAG):
    with open(_RESTART_FLAG, "w"):
        pass
    print("Install complete - restarting the runtime once so NumPy loads cleanly.")
    print("After it reconnects, run the cells again (Runtime > Run all).")
    os.kill(os.getpid(), 9)

In [ ]:
# Ensure Plotly figures render in Google Colab (a no-op in other notebook frontends).
import plotly.io as pio

try:
    import google.colab  # noqa: F401  (present only on Colab)

    pio.renderers.default = "colab"
except ImportError:
    pass

This page is two things at once: an **extended user guide** for
`analyze_convergence_clubs` — what it does, every argument, and everything it returns — and a
**testing environment** that generates synthetic data with a *known* club structure and checks
that the function recovers it. If a cell's `assert` ever fails, the function is broken.

## What is club convergence?

β- and σ-convergence assume the panel is *one* group heading toward *one* path. Often it is not:
some economies catch up to a high path while others settle on a lower one. **Club convergence**
(Phillips & Sul, 2007/2009) tests for exactly that — *multiple* convergence equilibria — without
grouping units in advance.

Each unit is modelled as a common trend $\mu_t$ scaled by a time-varying, unit-specific loading,
$X_{it} = \delta_{it}\,\mu_t$. Removing the common trend gives the **relative transition path**

$$
h_{it} = \frac{X_{it}}{\frac1N\sum_i X_{it}},
$$

whose cross-sectional mean is 1 by construction. If the units converge, the cross-sectional
variance $H_t = \frac1N\sum_i (h_{it}-1)^2 \to 0$, and the **log(t) regression**

$$
\log\!\left(\frac{H_1}{H_t}\right) - 2\log(\log t) = a + b\,\log t + \varepsilon_t,
\qquad t = [rT], \dots, T,
$$

has a non-negative slope $b = 2\alpha$. A one-sided $t_b > -1.65$ **fails to reject** convergence.
When the whole panel rejects, a **data-driven clustering algorithm** sorts units by their final
level, forms a core group by maximising $t_b$, sieves in the units that keep the group
converging, recurses on the residual, and finally **merges** adjacent clubs that jointly
converge. The series is first smoothed with the **Hodrick–Prescott filter** ($\lambda=400$ for
annual data) so the test runs on the long-run trend rather than the business cycle.

This is a faithful port of the Stata `psecta` package (Du, 2017); the log(t) statistic uses the
Phillips–Sul scalar long-run-variance HAC (Andrews 1991 quadratic-spectral kernel, AR(1)
automatic bandwidth), reproduced in NumPy. The variable is used **as you supply it** — pass `log`
GDP per capita / log labor productivity.

In [ ]:
import numpy as np
import pandas as pd

import expdpy as ex

## 1. The method in one cell

`analyze_convergence_clubs(df, var, ...)` needs only the variable and the panel ids. Here is the
clustering of (log) GDP per capita across countries in the bundled `productivity` panel — a
balanced 108-country, 25-year Penn World Table extract:

In [ ]:
from expdpy.data import load_productivity

prod = load_productivity()
res = ex.analyze_convergence_clubs(prod, "log_gdppc", entity="country", time="year")
res.fig

Each line is a club's **average** relative transition path; the dashed line at 1 is the
cross-sectional mean. `res.interpret()` reads the result in plain language:

In [ ]:
print(res.interpret())

## 2. How the function works

### Arguments

| argument | what it does | when to change it |
|---|---|---|
| `var` | the panel variable analysed (used **as-is**, no log) | pass `log` GDP per capita / log labor productivity |
| `entity`, `time` | the panel ids | omit if declared once via `set_panel` |
| `filter` | `"hp"` (default) HP-filters each unit and analyses the trend; `None` uses the variable as given | `None` if you pass an already-detrended series |
| `hp_lambda` | HP smoothing parameter | `400` for annual data (the literature default) |
| `r` | log(t) trimming fraction — the first `round(r*T)` periods are dropped | `0.3` for moderate `T`, `0.2` for large `T` |
| `method` | within-club sieve: `"adjust"` (Schnurbus 2016) or `"ps"` (original Phillips–Sul) | `"ps"` to reproduce the original 2007 rule |
| `merge` | adjacent-club merging: `"iterative"` / `"single"` / `"none"` | `"none"` to inspect the raw clusters |
| `fr` | sort by the last period (`0`) or the mean of the last `(1-fr)` periods | `fr>0` when endpoints are noisy |

### Everything it returns

In [ ]:
print(
    "clubs    :",
    res.n_clubs,
    "| divergent:",
    res.n_divergent,
    "| global t:",
    round(res.global_tstat, 2),
    "| converged:",
    res.converged,
)
print("figures  : fig, fig_paths, fig_clubs")
print("frames   : df", list(res.df.columns), "| summary", list(res.summary.columns))
res.glance()

The classification table `gt` (and the `summary` frame behind it) lists each club's size, its
log(t) slope `b` and t-statistic, and its members:

In [ ]:
res.gt

`membership` is the tidy `entity -> club` appendix list, and `fig_paths` / `fig_clubs` show every
unit's path coloured by club and a per-club small-multiples panel:

In [ ]:
res.fig_clubs

## 3. Does it recover the truth?

The cleanest test **plants** a known club structure: every unit in club $k$ sits at a distinct
long-run level $\mu_k$ plus an idiosyncratic deviation that decays geometrically, so units within
a club converge to a common path while the distinct levels keep the panel from converging
globally. The algorithm should reject global convergence and recover the planted partition.

In [ ]:
def club_panel(
    *,
    n_per_club=12,
    levels=(10.0, 9.3, 8.6),
    n_years=35,
    rho=0.9,
    spread=0.4,
    noise=0.002,
    seed=0,
):
    """Panel with planted clubs; returns (df, {unit: true_club})."""
    rng = np.random.default_rng(seed)
    rows, truth = [], {}
    for k, mu in enumerate(levels, start=1):
        for j in range(n_per_club):
            uid = f"c{k}u{j:02d}"
            truth[uid] = k
            dev = float(rng.uniform(-spread, spread))
            for t in range(1, n_years + 1):
                rows.append(
                    (uid, t, mu + dev * rho ** (t - 1) + float(rng.normal(0, noise)))
                )
    return pd.DataFrame(rows, columns=["unit", "year", "x"]), truth


panel, truth = club_panel(seed=1)
fit = ex.analyze_convergence_clubs(panel, "x", entity="unit", time="year")
print(f"global log(t) t = {fit.global_tstat:.2f}  ->  converged = {fit.converged}")
print(f"detected clubs  = {fit.n_clubs}  (planted 3)")
fit.summary[["club", "n_members", "beta", "tstat", "converging"]]

In [ ]:
# Best-match accuracy: each detected club scored by its modal planted club.
detected = dict(zip(fit.membership["entity"], fit.membership["club"], strict=True))
by_det = {}
for uid, det in detected.items():
    by_det.setdefault(int(det), []).append(truth[uid])
correct = sum(
    sum(1 for tc in trues if tc == max(set(trues), key=trues.count))
    for det, trues in by_det.items()
    if det != 0
)
accuracy = correct / len(truth)

assert fit.converged is False  # distinct levels => no global convergence
assert fit.global_tstat <= -1.65
assert fit.n_clubs == 3  # the three planted clubs
assert accuracy == 1.0  # every unit in its true club
print(f"recovered {fit.n_clubs} clubs, {accuracy:.0%} of units correctly placed")

When the panel really is one group — all units converging to a single level — the whole-panel
test should *fail to reject* and return a single club:

In [ ]:
one, _ = club_panel(levels=(10.0,), n_per_club=40, seed=2)
solo = ex.analyze_convergence_clubs(one, "x", entity="unit", time="year")
assert solo.converged is True and solo.n_clubs == 1
print(f"single converging group: converged={solo.converged}, clubs={solo.n_clubs}")

## 4. Convergence clubs across countries

Back to real data. Across the 108 countries, whole-panel convergence is firmly **rejected** and
the algorithm splits the world into several catch-up clubs plus a divergent tail — the textbook
"multiple equilibria" picture:

In [ ]:
print(res.interpret())

In [ ]:
# Each reported club clears the convergence threshold.
clubs = res.summary[res.summary["club"] != "Divergent"]
assert bool((clubs["tstat"] > -1.65).all())
res.summary[["club", "n_members", "tstat", "converging"]]

The same call works on log **labor productivity** (`log_lp`) — the other variable in the panel —
and on any balanced panel of your own.

## See also

- `ex.learn_convergence_clubs()` — a runnable Learn sandbox that plants a known club structure
  and shows the algorithm recover it.
- `ex.explain("convergence_clubs")` — the concept explainer (also `res.explain()`).
- `analyze_beta_convergence` / `analyze_sigma_convergence` — the single-group convergence views.

In [ ]:
ex.explain("convergence_clubs")